# 🛒 Nexora — Exploratory Data Analysis: Global Dynamics & Temporal Patterns
**Phase 4: Multi-Level EDA | Notebook 01**

### Objective
Understand global order volume dynamics, temporal purchasing patterns (hour of day, day of week), repurchase intervals, and shopping basket distributions across 3.4M+ orders.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling setup
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

DATA_DIR = Path("../data/processed")
print("Loading cleaned dataset tables...")
df_orders = pd.read_csv(DATA_DIR / "orders.csv")
df_products = pd.read_csv(DATA_DIR / "products.csv")
print(f"Loaded {len(df_orders):,} orders and {len(df_products):,} products.")


---
## ❓ Business Question 1: What are the peak ordering days and hours for consumers?
*Understanding peak demand periods is critical for server load balancing, delivery logistics scheduling, and targeted ad bidding.*


In [ ]:
# 1. Day of week and Hour of day analysis
dow_map = {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday'}
df_orders['dow_name'] = df_orders['order_dow'].map(dow_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day of week volume
dow_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
sns.countplot(data=df_orders, x='dow_name', order=dow_order, ax=axes[0], color='#2b5c8f')
axes[0].set_title("Order Volume by Day of Week", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Day of Week")
axes[0].set_ylabel("Total Orders")

# Hour of day volume
sns.countplot(data=df_orders, x='order_hour_of_day', ax=axes[1], color='#e26d5c')
axes[1].set_title("Order Volume by Hour of Day", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Hour of Day (0-23)")
axes[1].set_ylabel("Total Orders")

plt.tight_layout()
plt.show()


In [ ]:
# 2. Day-Hour Heatmap
pivot_time = df_orders.pivot_table(index='order_dow', columns='order_hour_of_day', values='order_id', aggfunc='count')
pivot_time.index = [dow_map[i] for i in pivot_time.index]

plt.figure(figsize=(12, 5))
sns.heatmap(pivot_time, cmap="YlGnBu", cbar_kws={'label': 'Order Count'})
plt.title("Consumer Order Intensity: Day of Week vs. Hour of Day", fontsize=14, fontweight='bold')
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.show()


---
## ❓ Business Question 2: What is the repurchase cadence and replenishment cycle?
*Identifying customer replenishment intervals (e.g. 7-day, 14-day, 30-day spikes) enables automated reorder reminders and smart cart push notifications.*


In [ ]:
# Repurchase interval distribution (excluding initial first orders where days=0 & is_first_order=1)
repeat_orders = df_orders[df_orders['is_first_order'] == 0]

plt.figure(figsize=(12, 5))
sns.histplot(repeat_orders['days_since_prior_order'], bins=31, color='#38b000', discrete=True, kde=False)
plt.title("Distribution of Days Between Consecutive Purchases", fontsize=14, fontweight='bold')
plt.xlabel("Days Since Prior Order")
plt.ylabel("Number of Orders")
plt.axvline(x=7, color='red', linestyle='--', label='Weekly Cycle (Day 7)')
plt.axvline(x=14, color='orange', linestyle='--', label='Bi-Weekly Cycle (Day 14)')
plt.axvline(x=30, color='purple', linestyle='--', label='Monthly Cap / Cap Spike (Day 30)')
plt.legend()
plt.show()


---
## ❓ Business Question 3: How large are shopping baskets across transactions?
*Understanding basket size distribution informs minimum order thresholds, free shipping tiers, and bundling opportunities.*


In [ ]:
# Sample order products to inspect basket size distribution
df_op_sample = pd.read_csv(DATA_DIR / "order_products.csv", nrows=3000000)
basket_sizes = df_op_sample.groupby("order_id").size()

print("--- Basket Size Summary Statistics ---")
print(f"Mean Basket Size:   {basket_sizes.mean():.2f} items")
print(f"Median Basket Size: {basket_sizes.median():.0f} items")
print(f"75th Percentile:    {basket_sizes.quantile(0.75):.0f} items")
print(f"95th Percentile:    {basket_sizes.quantile(0.95):.0f} items")

plt.figure(figsize=(11, 4))
sns.histplot(basket_sizes, bins=40, color='#6a040f', binrange=(1, 40))
plt.title("Distribution of Basket Sizes (Items per Order)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Items in Basket")
plt.ylabel("Frequency")
plt.axvline(x=basket_sizes.median(), color='blue', linestyle='--', label=f'Median = {basket_sizes.median():.0f} items')
plt.legend()
plt.show()


---
## ❓ Business Question 4: What is the overall reorder ratio across transactions?
*Reorder ratio reflects brand stickiness, consumable necessity vs novelty, and platform retention.*


In [ ]:
# Reorder proportion
reorder_rate = df_op_sample['reordered'].mean()
print(f"Overall Product Reorder Rate: {reorder_rate * 100:.2f}%")

plt.figure(figsize=(6, 4))
sns.barplot(x=['First-Time Purchase', 'Reordered Item'], y=[1 - reorder_rate, reorder_rate], palette=['#457b9d', '#e63946'])
plt.title("Item Purchase Type Breakdown", fontsize=13, fontweight='bold')
plt.ylabel("Proportion")
plt.ylim(0, 1.0)
for i, v in enumerate([1 - reorder_rate, reorder_rate]):
    plt.text(i, v + 0.03, f"{v*100:.1f}%", ha='center', fontweight='bold')
plt.show()
